<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2025</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Cython</b><br/>
    <br/>
    <font>Ислам Абасов</font><br/>
</center>

# [CYTHON](https://habr.com/ru/companies/ruvds/articles/462487/)

Cython — это язык программирования, который является надстройкой над Python и C. Он предназначен для упрощения написания C-расширений для Python и оптимизации Python-кода для повышения производительности. Cython позволяет вам добавлять статическую типизацию и напрямую вызывать функции C, что делает его идеальным инструментом для повышения скорости выполнения кода, написанного на Python.

### Основные характеристики Cython:

1. **Улучшение производительности**:
   - Cython позволяет компилировать Python-код в C, что может значительно ускорить его выполнение. Вы можете добавлять статические типы переменных, что упрощает оптимизацию и делает код более производительным.

2. **Простота интеграции с C/C++**:
   - Cython упрощает взаимодействие с C/C++ библиотеками. Вы можете легко вызывать функции C и использовать структуры данных, что позволяет интегрировать сторонние библиотеки в ваши Python-приложения.

3. **Простота написания и отладки**:
   - Cython синтаксически подобен Python, что делает его простым для понимания. Вы можете использовать привычный синтаксис Python, добавляя при этом элементы, которые позволяют улучшить производительность.

4. **Поддержка Cython-библиотек**:
   - Существует множество библиотек, написанных на Cython, которые могут быть использованы для ускорения различных задач. Это включает в себя научные библиотеки, такие как NumPy и SciPy.

## Типы в Cython

При использовании Cython можно выделить два набора типов. Один — для переменных, второй — для функций.

Если речь идёт о переменных, то тут нам доступны следующие типы:

* cdef int a, b, c
* cdef char *s
* cdef float x = 0.5 (число одинарной точности)
* cdef double x = 63.4 (число двойной точности)
* cdef list names
* cdef dict goals_for_each_play
* cdef object card_deck


При работе с функциями нам доступны следующие типы:

* def — обычная Python-функция, вызывается только из Python.
* cdef — Cython-функция, которую нельзя вызвать из обычного Python-кода. Такие функции можно вызывать только в пределах Cython-кода.
* cpdef — Функция, доступ к которой можно получить и из C, и из Python.

In [1]:
!pip3 install cython

In [2]:
%%writefile cython_range.pyx

cdef int GLOBAL = 42

cpdef int get_global():
    return GLOBAL

cpdef int test(int x):
    cdef int y = 1
    cdef int i
    for i in range(1, x+1):
        y *= i
    return y

Writing cython_range.pyx


In [3]:
%%writefile python_range.py

def test(x):
    y = 1
    for i in range(1, x+1):
        y *= i
    return y

Writing python_range.py


In [1]:
%%writefile setup.py

from distutils.core import setup
from Cython.Build import cythonize

setup(ext_modules = cythonize('cython_range.pyx'))

Overwriting setup.py


In [2]:
!python3 setup.py build_ext --inplace

In [8]:
import cython_range
import python_range
import time

number = 10000

start = time.time()
cython_range.test(number)
end =  time.time()

py_time = end - start
print("Cython time = {}".format(py_time))

start = time.time()
python_range.test(number)
end =  time.time()

cy_time = end - start
print("Python time = {}".format(cy_time))

print("Speedup = {}".format(cy_time / py_time))

Cython time = 6.508827209472656e-05
Python time = 0.04636883735656738
Speedup = 712.3992673992674


In [9]:
cython_range.get_global()

42

In [10]:
%%writefile csrc/mymath.h

#ifndef MYMATH_H
#define MYMATH_H

#include <stddef.h>

int add(int a, int b);
long long sum_array(const int* a, size_t n);

#endif // MYMATH_H

Overwriting csrc/mymath.h


In [11]:
%%writefile csrc/mymath.c

#include "mymath.h"

int add(int a, int b) {
    return a + b;
}

long long sum_array(const int* a, size_t n) {
    long long sum = 0;
    for (size_t i = 0; i < n; ++i) {
        sum += a[i];
    }
    return sum;
}

Overwriting csrc/mymath.c


In [12]:
%%writefile mymath.pyx

cdef extern from "mymath.h":
    int add(int a, int b)
    long long sum_array(const int* a, size_t n)

cpdef int add_py(int a, int b):
    return add(a, b)

def sum_array_py(int[::1] arr):
    cdef Py_ssize_t n = arr.shape[0]
    return sum_array(&arr[0], <size_t>n)

Writing mymath.pyx


In [12]:
%%writefile setup.py

from setuptools import setup, Extension
from Cython.Build import cythonize
import os

extensions = [
    Extension(
        name="mymath",
        sources=["mymath.pyx", os.path.join("csrc", "mymath.c")],
        include_dirs=["csrc"],  # чтобы найти mymath.h
        language="c",
    )
]

setup(
    name="mymath",
    version="0.0.1",
    ext_modules=cythonize(extensions, language_level=3),
)

Overwriting setup.py


In [13]:
!python setup.py build_ext -i

running build_ext
building 'mymath' extension
cc -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -g -O3 -Wall -arch arm64 -mmacosx-version-min=11.0 -Wno-nullability-completeness -Wno-expansion-to-defined -Wno-undef-prefix -fPIC -Icsrc -I/Users/i.g.abasov/prog/aba/shad_env/include -I/Users/i.g.abasov/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/include/python3.13 -c csrc/mymath.c -o build/temp.macosx-11.0-arm64-cpython-313/csrc/mymath.o
cc -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -g -O3 -Wall -arch arm64 -mmacosx-version-min=11.0 -Wno-nullability-completeness -Wno-expansion-to-defined -Wno-undef-prefix -fPIC -Icsrc -I/Users/i.g.abasov/prog/aba/shad_env/include -I/Users/i.g.abasov/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/include/python3.13 -c mymath.c -o build/temp.macosx-11.0-arm64-cpython-313/mymath.o
cc -bundle -undefined dynamic_lookup -arch arm64 -mmacosx-version-min=11.0 -LModules/_hacl -Wl,-headerpad,40 build/temp.

In [14]:
import mymath
print(mymath.add_py(2, 3))

5


In [15]:
import numpy as np, mymath
a = np.array([1, 2, 3, 4], dtype=np.int32)
print(mymath.sum_array_py(a))

10


### Основные преимущества Cython:

- **Быстродействие**: С помощью Cython код может работать намного быстрее, чем чистый Python, когда он правильно оптимизирован. Статическая типизация и компиляция в C позволяют значительно уменьшить накладные расходы.
  
- **Совместимость**: Cython полностью совместим с Python и его библиотеками. Это позволяет плавно интегрировать существующий Python-код в Cython без необходимости переписывать его.

- **Улучшенная отладка**: Cython поддерживает отладку, что позволяет разработчикам находить и устранять проблемы, даже если код компилируется в C.

### Ограничения Cython:

- **Сложность при написании**: Хотя Cython имеет синтаксис, похожий на Python, прежде всего, необходимо обеспечить правильную типизацию и управление памятью, что может добавить сложности разработке.
  
- **Необходимость компиляции**: Cython-код нужно компилировать, что требует дополнительно настроенной среды для сборки и может привести к усложнению процессов разработки и деплоя.

### [pybind11](https://github.com/pybind/pybind11)

Pybind11 - заголовочная C++ библиотека для создания Python-расширений

Что он делает
* экспортирует C++ функции, классы, перечисления и константы в Python
* конвертирует стандартные типы
* интегрируется с numpy
* управляет GIL

In [21]:
%%writefile module.cpp

#include <pybind11/pybind11.h>
#include <cmath>
#include <string>

namespace py = pybind11;

// Свободные функции
int add(int a, int b) { return a + b; }
std::string greet(const std::string& name) { return "Hello, " + name + "!"; }

struct Point {
    double x, y;
    Point(double x_, double y_) : x(x_), y(y_) {}
    double length() const { return std::sqrt(x * x + y * y); }
    void move(double dx, double dy) { x += dx; y += dy; }
};


PYBIND11_MODULE(simplebind, m) {
    m.doc() = "док стринга";

    m.def("add", &add, "Сложение двух целых");
    m.def("greet", &greet, py::arg("name"), "Приветствие");

    py::class_<Point>(m, "Point")
        .def(py::init<double,double>(), py::arg("x"), py::arg("y"))
        .def_readwrite("x", &Point::x)
        .def_readwrite("y", &Point::y)
        .def("length", &Point::length, "Длина вектора")
        .def("move", &Point::move, py::arg("dx"), py::arg("dy"), "Сдвиг точки");
}

Writing module.cpp


In [16]:
%%writefile setup.py

from setuptools import setup
from pybind11.setup_helpers import Pybind11Extension, build_ext

ext_modules = [
    Pybind11Extension(
        "simplebind",         
        ["module.cpp"],        
    )
]

setup(
    name="simplebind",
    version="0.0.1",
    description="Minimal pybind11 example",
    ext_modules=ext_modules,
    cmdclass={"build_ext": build_ext}, 
)

Overwriting setup.py


In [17]:
!python setup.py build_ext -i

running build_ext
c++ -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -g -O3 -Wall -arch arm64 -mmacosx-version-min=11.0 -Wno-nullability-completeness -Wno-expansion-to-defined -Wno-undef-prefix -fPIC -I/Users/i.g.abasov/prog/aba/shad_env/include -I/Users/i.g.abasov/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/include/python3.13 -c flagcheck.cpp -o flagcheck.o -std=c++17
copying build/lib.macosx-11.0-arm64-cpython-313/simplebind.cpython-313-darwin.so -> 


In [18]:
import simplebind as sb

print(sb.add(2, 3))       
print(sb.greet("PyBind")) 

p = sb.Point(3, 4)
print(p.length())        
p.move(1, -2)
print(p.x, p.y)         

5
Hello, PyBind!
5.0
4.0 2.0


Плюсы:
* удобно биндить классы, методы, исключения, шаблоны
* управление GIL
* кроссплатформенная сборка
* прозрачные сигнатуры и ошибки
* можно писать на C++

Минусы
* требует C++ компилятор
* большое время компиляции, "шумные" ошибки
* надо писать на C++